# Workflow: Xenium {#sec-img-workflow-xenium}



## Preamble

### Introduction

In this demo, we will analyze a 313-plex Xenium dataset on human colorectal cancer tissue
[@Janesick2023-high-res]. Following very basic quality control and preprocessing, we will 
perform both (non-spatial) unsupervised clustering as well as fully supervised 
label-transfer based on scRNA-seq reference data, and then compare the obtained 
cluster assignments to those provided by the authors.

### Dependencies {#sec-img-workflow-xenium-load-data}

In [ ]:
library(BiocParallel)
library(DropletUtils)
library(ggplot2)
library(OSTA.data)
library(patchwork)
library(scater)
library(scrapper)
library(SingleR)
library(SpatialExperiment)
library(SpatialExperimentIO)
# set parallelization
bp <- MulticoreParam(th <- 4)
# set seed for random number generation
# in order to make results reproducible
set.seed(112358)

In [ ]:
# retrieve dataset from OSF repository
id <- "Xenium_HumanColon_Oliveira"
pa <- OSTA.data_load(id, mol=FALSE)
dir.create(td <- tempfile())
unzip(pa, exdir=td)
(spe <- readXeniumSXE(td, addTx=FALSE))

In [ ]:
#| code-fold: true
.plt_xy <- \(spe, col) {
    df <- data.frame(colData(spe), spatialCoords(spe))
    aes <- if (is.numeric(df[[col]])) {
        theme(
            legend.key.height=unit(1, "lines"), 
            legend.key.width=unit(0.5, "lines"))
    } else {
        list(
            theme(legend.key.size=unit(0, "lines")),
            guides(col=guide_legend(override.aes=list(alpha=1, size=2))))
    }
    ggplot(df, aes(x_centroid, y_centroid, col=.data[[col]])) +
        coord_equal() + theme_void() + aes +
        geom_point(stroke=0, size=1/3)
} 

## Quality control

In [ ]:
# compute cell-level QC metrics &
# identify low-quality cells by thresholding on
# median absolute deviation (MAD) from the median
spe <- quickRnaQc.se(spe, subsets=list())
# tabulate # and % of cells discarded 
# due to few counts/detected features
round(100*prop.table(table(keep=!spe$keep)), 2)

Before proceeding to exclude any cells from downstream analyses, let's 
first visualize the cells deemed to be of high quality in space alongside 
the underlying quality control metrics (total counts and detected features):

In [ ]:
#| code-fold: true
.plt_xy(spe, "sum") +  
    scale_color_viridis_c(
        "# counts", 
        trans="log1p",
        breaks=range(spe$sum), 
        labels=c("low", "high")) +
.plt_xy(spe, "detected") +
    scale_color_viridis_c(
        "# features", 
        breaks=range(spe$detected), 
        labels=c("low", "high")) +
.plt_xy(spe[, order(!spe$keep)], "keep") + 
    scale_color_manual(
        "high-quality",
        labels=c("no", "yes"),
        values=c("purple", "lavender"))

In [ ]:
# discard low-quality cells
ncol(spe <- spe[, spe$keep])

## Processing

For the sake of runtime, we will perform downstream analyses only on
a square crop of the tissue, defined by the following px coordinates:

In [ ]:
box <- list(xmin=2e3, xmax=5e3, ymin=1e3, ymax=4e3)

Cropping to this region, we retain fewer than 90,000 cells:

In [ ]:
xy <- spatialCoords(spe)
i <- 
    xy[, 1] > box$xmin &
    xy[, 1] < box$xmax &
    xy[, 2] > box$ymin &
    xy[, 2] < box$ymax 
ncol(sub <- spe[, i])

In [ ]:
#| code-fold: true
df <- data.frame(xy, i)
p <- ggplot(df, 
    aes(x_centroid, y_centroid)) +
    coord_equal() + theme_void() + 
    theme(legend.position="none")
p + geom_point(aes(col=i), stroke=0, size=0.1) |
p + geom_point(data=df[i, ], stroke=0, size=0.2)

Next, we'll log-normalize counts by area, and perform principal 
component analysis (PCA) on all `r nrow(spe)` RNA targets:

In [ ]:
# cell area-based normalization
sfs <- (. <- sub$cell_area) / median(.)
sub <- normalizeRnaCounts.se(sub, size.factors=sfs)
# principal component analysis
sub <- runPca.se(sub, features=rownames(sub))

Let's visualize the expression of some genes in space; e.g., PIGR, IGHG3 and 
CEACAM6, which should mark epithelial, plasma and tumor cells, respectively:

In [ ]:
#| code-fold: true
gs <- c("PIGR", "IGHG3", "CEACAM6")
es <- scale(logcounts(sub))
es <- t(as.matrix(es[gs, ]))
colData(sub) <- cbind(colData(sub), es)
ps <- lapply(gs, \(.) .plt_xy(sub, .) + ggtitle(.))
wrap_plots(ps, nrow=1) &
    scale_color_gradientn(NULL, 
        labels=c("low", "high"), 
        colors=rev(hcl.colors(9, "PuRd")),
        limits=rng, breaks=rng <- range(es)) & 
    theme(plot.title=element_text(hjust=0.5))

## Annotation

### Unsupervised

In [ ]:
# shared nearest-neighbor (SNN) graph based on 
# cell-to-cell Jaccard similarity in PC space;
# community detection using Leiden algorithm
sub <- clusterGraph.se(sub,
    num.threads=th, resolution=0.5, 
    method="leiden", output.name="Leiden",
    more.build.args=list(weight.scheme="jaccard"))
table(sub$Leiden)

### Supervised

For comparison, we annotate the Xenium data using a label transfer approach, 
`r BiocStyle::Biocpkg("SingleR")`, which relies on labeled scRNA-seq data to 
compute references profiles and transfers labels based on the rank correlation 
between observed (here, Xenium) and reference (scRNA-seq) expression profiles.

First, we retrieve a matching (Chromium) scRNA-seq dataset, which includes low- 
(`Level1`) and high-resolution (`Level2`) annotations of cells into 9 and 31 
subpopulations, respectively:

In [ ]:
# retrieve dataset from OSF repository
id <- "Chromium_HumanColon_Oliveira"
pa <- OSTA.data_load(id)
dir.create(td <- tempfile())
unzip(pa, exdir=td)

# read into 'SingleCellExperiment'
sce <- read10xCounts(list.files(td, "h5$", full.names=TRUE))
cd <- read.csv(list.files(td, "cell_meta", full.names=TRUE))
colData(sce) <- cbind(colData(sce), cd[, -1])
table(sce$Level1) # tabulate low-res. labels
ncol(sce) # overall number of cells

[Note that we filter the reference data to contain only cells from the same patient. This is not strictly necessary, assuming that clusters are transcriptionally stable across patients, but is done here to reduce runtime.]{.aside}
Here, we run `SingleR` using `Level2` (high-resolution) annotations and with argument
`aggr.ref=TRUE`, such that reference profiles will be aggregated (per cluster)
prior to annotation. In this way, every Xenium cell will be assigned a
label based on which pseudo-bulk scRNA-seq profile represents the best match.

In [ ]:
# exclude cells deemed to be of low-quality
sce <- sce[, sce$QCFilter == "Keep"]
# subset cells from same patient
sce <- sce[, grepl("P2", sce$Patient)]
# realize count matrix
assay(sce) <- as(assay(sce), "dgCMatrix")
# log-library size normalization
sce <- normalizeRnaCounts.se(sce)
# restrict to Xenium targets
sce <- sce[rowData(sce)$ID %in% rowData(sub)$ID, ]
# set gene symbols as feature names
rownames(sce) <- rowData(sce)$Symbol
# perform label transfer at the single cell-level,
# using pseudo-bulk Chromium profiles as reference
res <- SingleR(
    test=sub, ref=sce, 
    labels=sce$Level2, 
    de.method="wilcox",
    aggr.ref=TRUE, BPPARAM=bp)
sub$Level2 <- factor(res$pruned.labels)

Based on these predictions, we can also propagate `Level1` (low-resolution) annotations:

In [ ]:
idx <- match(sub$Level2, sce$Level2)
table(sub$Level1 <- factor(sce$Level1[idx]))

Simplifying further, we can group cells into different compartments, namely,
(malignant) tumor, immune, epithelial and stromal cells; we'll see below that
visualizing cells in this way nicely captures the general tissue structure.

In [ ]:
lab <- list(
    tum=c("Tumor"),
    epi=c("Intestinal Epithelial"),
    imm=c("B cells", "T cells", "Myeloid"),
    str=c("Endothelial", "Fibroblast", "Smooth Muscle"))
idx <- match(sub$Level1, unlist(lab))
lab <- rep.int(names(lab), sapply(lab, length))
table(sub$Level0 <- factor(lab[idx]))

### Comparison

Tabulating the cluster assignments between Leiden (unsupervised) and `SingleR` 
(supervised), we can observe overall high concordance; i.e., most clusters have
a one-to-one mapping between both approaches. However, some subpopulations are
split between clusters; e.g., cells labeled as cluster fibroblasts, endothelia 
and smooth muscle cells by `SingleR` tend to intermix in the Leiden clusters. 
This is not unexpected, given that these are all stromal subpopulations with 
comparatively similar transcriptional profiles. (Note that we are observing a
mere fraction of the whole transcriptome with the Xenium panel employed here.)

In [ ]:
# contingency table & number of clusters
round(100*prop.table(table(sub$Level1, sub$Leiden), 2), 1)
c(SingleR=nlevels(sub$Level1), Leiden=nlevels(sub$Leiden))

In [ ]:
#| code-fold: true
lapply(c("Leiden", "Level0", "Level1"), \(.) {
    pal <- if (. == "Level0") {
        c("gold", "cyan", "magenta", "black")
    } else {
        hcl.colors(nlevels(sub[[.]]), "Spectral")
    }
    .plt_xy(sub, .) + scale_color_manual(values=pal) 
}) |> wrap_plots()

## Downstream

### Marker genes

Below, we test for differential expression between `Level1` clusters, and 
visualize selected markers as a heatmap of (z-scaled) average expression.
It's comforting to see that we pick up on many classics, e.g., endothelia 
are marked by VWF and PECAM1, T cells by CD2 and TRAC, etc.

In [ ]:
# test for differential expression between clusters
ok <- !is.na(sub$Level1)
de <- scoreMarkers.se(sub[, ok], sub$Level1[ok])

# select top-ranked genes for every cluster
gs <- lapply(de, \(df) head(rownames(df), 5))
gs <- unique(unlist(gs))

# visualize their average expression by cluster
plotGroupedHeatmap(sub, 
    features=gs, group="Level1", 
    scale=TRUE, center=TRUE, fontsize=6)

## Appendix

### References {.unnumbered}